# 04 — Dense Networks (MLP) Baseline

This notebook trains the simplest Deep Learning model of the study: the image is **flattened** and passed through fully connected (Dense) hidden layers ending in one **logit** per garment class. Softmax converts logits to probabilities for interpretation; it is not applied before PyTorch CrossEntropyLoss. It reproduces the "Image: Dense Layers (only)" experiment of the course guidelines on our H&M dataset.

Beyond accuracy, the goal is to understand the basics of training a deep feed-forward network: input normalisation, activation functions, cross-entropy loss, optimiser, batch size, epochs, regularisation and the behaviour of training and validation curves. It also shows why ignoring spatial structure limits Dense networks on images.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Vía A · Arquitectura básica desde cero

Este es el primer experimento de la vía scratch interpretada a partir de los apuntes de clase: pesos aleatorios, píxeles aplanados y un clasificador Dense. El objetivo es comprender pérdida, activaciones, gradientes y regularización. No se busca ganar a las demás arquitecturas.

La salida usada por la pérdida será logits; `CrossEntropyLoss` ya incorpora la operación necesaria para la log-verosimilitud. Softmax se aplicará cuando se necesiten probabilidades, evitando aplicarlo dos veces. Compararemos una modificación cada vez y documentaremos incluso resultados negativos. Flatten + Dense no introduce por sí solo embeddings de palabras ni generación.


## Protocolo revisado antes del entrenamiento · 20/11/2026

La referencia común es `config/experimental_protocol_v1.json`. Es un diseño fijado antes de entrenar las redes, no una lista de resultados obtenidos. La versión equilibrada de 18.000 fotos se conserva como experimento exploratorio. Sus métricas describen las doce clases seleccionadas con ese muestreo; no estiman automáticamente el comportamiento sobre el catálogo original.

**Por defecto:** entropía cruzada sin pesos, sin aumentos y con normalización de train a la resolución de entrada. Pesos de clase y aumentos se activarán como ablaciones explícitas. Los pesos guardados en 02 son una opción calculada, no una instrucción de aplicarlos siempre. Cada experimento registrará resolución, preprocesamiento, IDs de productos, configuración, semilla y presupuesto.

**Métricas fijadas:** macro-F1 primaria; accuracy secundaria; precisión, recall, F1 y soporte por clase; matriz de confusión en recuentos y normalizada por filas. Comparar parámetros entrenables, tiempo de entrenamiento, latencia por imagen, memoria pico y hardware. Selección por macro-F1 de validation; early stopping por pérdida de validation; tres semillas (42, 123 y 2026). El test permanece cerrado hasta congelar decisiones; intervalos finales mediante remuestreo de grupos de producto, no imágenes independientes. Los baselines deterministas no necesitan simular tres entrenamientos diferentes.

### H1 · Dense sin una desventaja arbitraria por resolución

La referencia Dense será RGB de 64×64: 12.288 entradas, frente a 150.528 a 224×224. Con 256 neuronas en la primera capa, solo esa capa pasaría de 3.145.984 a 38.535.424 parámetros (incluidos sesgos). No usaremos el tamaño almacenado como argumento para imponer 224 a Dense.

Comparación controlada H1: Dense y CNN sobre las mismas imágenes RGB64, pérdida sin pesos, sin aumentos, mismas particiones y regla de parada. Definir previamente un objetivo de aproximadamente un millón de parámetros entrenables y tolerancia de ±10 %; si una arquitectura no puede cumplirlo, publicar el desajuste. Comparar además FLOPs estimados, tiempo y memoria: igualar parámetros no iguala coste. Una segunda comparación aplicada puede permitir resoluciones propias, pero no aislará únicamente la arquitectura.


## Objectives

- Implement a flatten + Dense + softmax classifier in PyTorch with the shared data pipeline.
- Use unweighted multi-class cross-entropy initially; test training-set class weights only as an explicit ablation and justify the output layer.
- Study the effect of depth, width, activation function, dropout and L2 on learning curves.
- Compare optimisers (SGD, SGD with momentum / Nesterov, Adam) and learning rates.
- Interpret the model by visualising first-hidden-layer weights as images and hidden activations.
- Report the Dense baseline (accuracy, macro F1, confusion matrix on validation) to compare with later models.

## Preview of the work

1. **Setup.** seeds, device, loaders at low resolution (e.g. 64×64 grayscale and RGB).
2. **Model definition.** `nn.Flatten` → hidden Dense layers → output logits; parameter count.
3. **Training loop.** loss, optimiser, mini-batches, early stopping on validation loss; training history saved.
4. **Learning curves.** loss and accuracy per epoch; under- / over-fitting diagnosis.
5. **Small controlled experiments.** depth, width, activation, dropout, optimiser and batch size, one change at a time.
6. **Interpretation.** first-layer weights as images, hidden-layer activations.
7. **Validation results and conclusions.** comparison with the classical baselines of notebook 03.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders and configuration from notebook 02 |
| **Outputs** | Training histories, metrics and best checkpoint in `results/` |
| **Course material** | `PRDL_MLLB_FeedForward.pptx`, `PRDL_MLLB_LogLikelihood_CrossEntropy_KLDivergence.pptx`, `MLLB_FeedForward_MNIST_in_Keras.ipynb`, `Pytorch_quickstart_tutorial_Fashion_MINIST.ipynb`, `nesterov_momentum_explained-v2.pptx` |